# Chapter 6: Fine-tuning for Classification
This chapter introduces fine-tuning approaches for pre-trained LLM models

#### 6.1 Different categories of fine-tuning
The most common ways of fine tuning models are instruction and classification. Instruction means that we train the model on a set of tasks using specific instructions e.g. translation. In classification fine-tuning the focus lies on classifying using a set of labels e.g. e-mail filtering with normal mail or spam mail labels. 

So the main difference is:
    - Classification: Restricted to predicting labels
    - Instruction: More general, can take on a broader range of tasks

### 6.2 Preparing the dataset
We need to prepare a dataset that the model can be fine-tuned on. 

In [1]:
# Import the dataset
import urllib.request
import zipfile
import os
from pathlib import Path

url            = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path       = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    """Function that donloads and unzips the spam data"""
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction")
        return
    # Download the file
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())
    # Unzip the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)
    
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File donwloaded and save as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction


In [2]:
# The data is now stored as a tab-seperated text file (tsv) which we can load into a pandas dataframe
import pandas as pd
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "text"])
df

,Label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [3]:
# Examine the label distribution
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [4]:
def create_balanced_dataset(df):
    """Function that handles an imbalanced dataset by undersampling"""
    num_spam   = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    return pd.concat([ham_subset, df[df["Label"] == "spam"]])

In [5]:
balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [6]:
# Convert labels into 1 & 0
balanced_df["Label"] = balanced_df["Label"].map({"ham":0, "spam":1})

In [7]:
def random_split(df, train_frac, validation_frac):
    """Function that takes in a dataframe and randomly shuffles it and then splits it 
       into three parts: training, validation and testing"""
    df = df.sample(frac=1, random_state=123).reset_index(drop=True) # Shuffle entire dataset
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end] # Training data
    validation_df = df[train_end:validation_end] # Validation data
    test_df = df[validation_end:] # Testing data

    return train_df, validation_df, test_df

In [8]:
# Split into 70% train, 10% validation and 20% test
traind_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

In [9]:
# Save files for later reuse
traind_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

### 6.3 Creating Data Loaders
Create new DataLoaders that works on the current data and our GPT model

In [10]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [ ]:
# Since all messages are different lengths we have to add padding to the shorter messages
# so that we can utilize a sliding-window approach
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    """The SpamDataset class finds longest sequence in dataset, encodes the text messages and pads messages where required"""

    def __init__(self, csv_file, tokenizer, max_length=None,
                 pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
                                            
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["text"]
        ]
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
                                           
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]
                                         
        self.encoded_texts = [
            encoded_text + [pad_token_id] * 
            (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]
    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )
    def __len__(self):
        return len(self.data)
    
    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

In [33]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

In [34]:
print(train_dataset.max_length)
# Model can handle sequences up to 1024 characters

120


In [35]:
validation_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

In [36]:
# Create dataloaders for training, validation and testing to use batches
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

val_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

In [37]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimension: ", input_batch.shape)
print("Label batch dimensions: ", target_batch.shape)

Input batch dimension:  torch.Size([8, 120])
Label batch dimensions:  torch.Size([8])


In [38]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


### 6.4 Initializing a model with pretrained weights
Before finetuning the model we need to prepare a pre-trained one if we wish to skip the training part

In [51]:
CHOOSE_MODEL  = "gpt2-small (124M)"
INPUT_PROMTPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate_embedding": 0.0,
    "drop_rate_multihead": 0.0,
    "drop_rate_shortcut": 0.0,
    "qkv_bias": True
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [53]:
import sys
sys.path.append("..")
from ch05.gpt_download import download_and_load_gpt2
from src.classes import GPTModel
from src.functions import *

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size, models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()
print(f"Model loaded succesfully")

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe
Model loaded succesfully


In [54]:
text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [ ]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

# Based on this output it's clear that the model can't yet follow instructions and just generates text

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner


### 6.5 Adding a classification head
Now that the model is loaded we will modify it for fine-tuning by replacing the original output layer to a layer mapping only to two classes: spam or not spam

In [57]:
# Let's take a look at the model architechture
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7